In [ ]:
import geopandas as gpd
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import pypsa

# 1. Load the solved or base network
network_path = "results/networks/base_s_40_elec_.nc"
n = pypsa.Network(network_path)

fig, ax = plt.subplots(figsize=(11, 10))

# 2. Add base country borders using Natural Earth GeoJSON
url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson"
world = gpd.read_file(url)
europe = world.cx[-15:35, 34:72]
europe.plot(ax=ax, color="#f3f4f6", edgecolor="#cbd5e1", linewidth=0.8)

# 3. Extract bus coordinates
bus_x = n.buses.x.to_dict()
bus_y = n.buses.y.to_dict()

# 4. Draw AC transmission lines
line_segments = [
    [(bus_x[b0], bus_y[b0]), (bus_x[b1], bus_y[b1])]
    for b0, b1 in zip(n.lines.bus0, n.lines.bus1)
    if b0 in bus_x and b1 in bus_x
]
line_coll = LineCollection(
    line_segments,
    colors="#475569",
    linewidths=1.2,
    alpha=0.7,
    zorder=2,
    label="AC Lines",
)
ax.add_collection(line_coll)

# 5. Draw DC transmission links (if present)
if not n.links.empty:
    link_segments = [
        [(bus_x[b0], bus_y[b0]), (bus_x[b1], bus_y[b1])]
        for b0, b1 in zip(n.links.bus0, n.links.bus1)
        if b0 in bus_x and b1 in bus_x
    ]
    link_coll = LineCollection(
        link_segments,
        colors="#2563eb",
        linewidths=1.2,
        linestyle="--",
        alpha=0.7,
        zorder=2,
        label="DC Links",
    )
    ax.add_collection(link_coll)

# 6. Plot network buses
ax.scatter(
    n.buses.x,
    n.buses.y,
    c="#0d9488",
    s=45,
    edgecolors="white",
    linewidth=0.8,
    zorder=3,
    label="Buses (40 clusters)",
)

# 7. Set bounding box around buses
margin = 2.0
ax.set_xlim(n.buses.x.min() - margin, n.buses.x.max() + margin)
ax.set_ylim(n.buses.y.min() - margin, n.buses.y.max() + margin)

# 8. Titles and labels
ax.set_title("PyPSA Electric Network Topology (40 Clusters)", fontsize=14, pad=12)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.grid(True, linestyle=":", alpha=0.4)
ax.legend(loc="upper left", frameon=True)

plt.tight_layout()
plt.show()